## 预计算非平稳深度域子波矩阵 `W_depth`

这个 notebook 聚焦一件事：把时间域子波 `w(t)`、深度采样轴 `z`、深度域速度 `v(z)` 组合成深度域非平稳子波矩阵：

`d_syn(z_l) = sum_j W_depth[l, j] * r_j`

默认示例用单井 `Vp` 曲线生成一条速度函数，轻量验证 `W_depth` 的构造、带状结构和正演效果。最后一个可选单元展示如何把速度输入替换成 `lfm_depth.py` 生成的井控低频速度模型。


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import lasio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.ndimage import gaussian_filter1d

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cup.petrel.load import extract_vp_log_from_las, import_well_heads_petrel

plt.rcParams["figure.dpi"] = 120
ROOT

## Paths and geometry

注意：这块 SEG-Y 的 sample header 虽然写成 time/ms，但这里按深度域解释：`dt=5 ms` 实际是 `dz=5 m`，`4750-7500 ms` 实际是 `4750-7500 m`。


In [ ]:
DATA = ROOT / "data"

SEISMIC_FILE = DATA / "raw" / "mero_84_coord_extend"
WELL_HEADS_FILE = DATA / "raw" / "well_heads"
LAS_DIR = DATA / "vertical_well_las_target_clear"
WAVELET_CSV = DATA / "output_auto_well_tie_wavelet_depth_tvdss_20260423" / "wavelet_201ms_2-ANP-2A-RJS.csv"

HORIZON_FILES = {
    "base_of_salt": DATA / "interpre_depth" / "base_of_salt_extend",
    "base_of_bve": DATA / "interpre_depth" / "base_of_bve_extend",
    "base_of_itp": DATA / "interpre_depth" / "base_of_itp_extend",
}

GEOMETRY = {
    "n_il": 601,
    "inline_min": 1501.0,
    "inline_max": 2101.0,
    "inline_step": 1.0,
    "n_xl": 801,
    "xline_min": 3799.0,
    "xline_max": 6999.0,
    "xline_step": 4.0,
    "n_sample": 551,
    "sample_min": 4750.0,
    "sample_max": 7500.0,
    "sample_step": 5.0,
    "sample_domain": "depth",
    "sample_unit": "m",
}

SEGY_OPTIONS = {"iline": 5, "xline": 21, "istep": 1, "xstep": 4}

SELECTED_WELL = "2-ANP-2A-RJS"
VP_MNEMONIC = "DTCO_QYZ_CLAER"
VP_UNIT = "us/m"

# 单井示例默认只取井曲线覆盖良好的窗口。生产版可替换为 Vp_lfm volume 的完整 trace。
DEPTH_WINDOW_M = (5000.0, 6030.0)
LOWPASS_SIGMA_M = 60.0

for path in [SEISMIC_FILE, WELL_HEADS_FILE, LAS_DIR, WAVELET_CSV, *HORIZON_FILES.values()]:
    if not Path(path).exists():
        raise FileNotFoundError(path)

GEOMETRY

## Load the time-domain wavelet

`wavelet_time_s` 已经以秒为单位保存，且包含负时延到正时延，因此后面直接按 `tau = twt(z_l) - twt(interface_j)` 做线性插值。超出子波时窗的权重置零。


In [ ]:
wavelet_df = pd.read_csv(WAVELET_CSV)
wavelet_time_s = wavelet_df["time_s"].to_numpy(dtype=np.float64)
wavelet_amp = wavelet_df["amplitude"].to_numpy(dtype=np.float64)

order = np.argsort(wavelet_time_s)
wavelet_time_s = wavelet_time_s[order]
wavelet_amp = wavelet_amp[order]

wavelet_dt_s = float(np.median(np.diff(wavelet_time_s)))
print(f"wavelet samples: {wavelet_amp.size}")
print(f"wavelet dt: {wavelet_dt_s * 1000:.3f} ms")
print(f"wavelet range: {wavelet_time_s[0] * 1000:.1f} to {wavelet_time_s[-1] * 1000:.1f} ms")

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.plot(wavelet_time_s * 1000.0, wavelet_amp, lw=1.5)
ax.axvline(0.0, color="k", lw=0.8, alpha=0.5)
ax.set_xlabel("time lag (ms)")
ax.set_ylabel("amplitude")
ax.set_title("Wavelet used by W_depth")
ax.grid(True, alpha=0.25);

## Core implementation

这里是核心。实现上故意先用 dense matrix，因为当前样点数 `N=551`，单道 `W_depth` 只有约 `551 x 550`，非常轻。后面可以转 CSR sparse 或在 batch 中按 trace 动态构建。


In [ ]:
def compute_twt_axes(depth_m: np.ndarray, velocity_mps: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return TWT at depth samples and impedance interfaces.

    depth_m and velocity_mps are sampled at impedance/depth sample centers.
    Reflectivity r[j] lives between sample j and j+1, so interface TWT is
    approximated as the midpoint TWT between neighboring sample centers.
    Absolute TWT origin is irrelevant because only TWT differences are used.
    """
    depth_m = np.asarray(depth_m, dtype=np.float64)
    velocity_mps = np.asarray(velocity_mps, dtype=np.float64)

    if depth_m.ndim != 1 or velocity_mps.ndim != 1:
        raise ValueError("depth_m and velocity_mps must be 1D arrays.")
    if depth_m.size != velocity_mps.size:
        raise ValueError(f"depth/velocity size mismatch: {depth_m.size} vs {velocity_mps.size}")
    if depth_m.size < 2:
        raise ValueError("At least two depth samples are required.")
    if np.any(np.diff(depth_m) <= 0.0):
        raise ValueError("depth_m must be strictly increasing.")
    if np.any(~np.isfinite(velocity_mps)) or np.any(velocity_mps <= 0.0):
        raise ValueError("velocity_mps must be finite and positive everywhere.")

    dz = np.diff(depth_m)
    slowness_mid = 0.5 * (1.0 / velocity_mps[:-1] + 1.0 / velocity_mps[1:])
    twt_sample_s = np.empty(depth_m.size, dtype=np.float64)
    twt_sample_s[0] = 0.0
    twt_sample_s[1:] = np.cumsum(2.0 * dz * slowness_mid)
    twt_interface_s = 0.5 * (twt_sample_s[:-1] + twt_sample_s[1:])
    return twt_sample_s, twt_interface_s


def build_w_depth_dense(
    depth_m: np.ndarray,
    velocity_mps: np.ndarray,
    wavelet_time_s: np.ndarray,
    wavelet_amp: np.ndarray,
    *,
    amplitude_threshold: float = 0.0,
) -> tuple[np.ndarray, dict]:
    """Build dense nonstationary depth-domain wavelet matrix.

    Returns W with shape (N_depth, N_depth - 1). It maps reflectivity located
    at impedance interfaces to seismic samples located at depth sample centers.
    """
    wavelet_time_s = np.asarray(wavelet_time_s, dtype=np.float64)
    wavelet_amp = np.asarray(wavelet_amp, dtype=np.float64)
    if wavelet_time_s.ndim != 1 or wavelet_amp.ndim != 1:
        raise ValueError("wavelet_time_s and wavelet_amp must be 1D arrays.")
    if wavelet_time_s.size != wavelet_amp.size:
        raise ValueError("wavelet_time_s and wavelet_amp must have the same length.")
    if np.any(np.diff(wavelet_time_s) <= 0.0):
        raise ValueError("wavelet_time_s must be strictly increasing.")

    twt_sample_s, twt_interface_s = compute_twt_axes(depth_m, velocity_mps)
    tau_s = twt_sample_s[:, None] - twt_interface_s[None, :]
    W = np.interp(tau_s.ravel(), wavelet_time_s, wavelet_amp, left=0.0, right=0.0)
    W = W.reshape(depth_m.size, depth_m.size - 1)

    if amplitude_threshold > 0.0:
        W[np.abs(W) < float(amplitude_threshold)] = 0.0

    meta = {
        "shape": W.shape,
        "n_nonzero": int(np.count_nonzero(W)),
        "density": float(np.count_nonzero(W) / W.size),
        "twt_sample_s": twt_sample_s,
        "twt_interface_s": twt_interface_s,
        "tau_min_s": float(np.nanmin(tau_s)),
        "tau_max_s": float(np.nanmax(tau_s)),
    }
    return W.astype(np.float32), meta


def reflectivity_from_impedance(ai: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    ai = np.asarray(ai, dtype=np.float64)
    return ((ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1] + eps)).astype(np.float32)


def summarize_w_depth(W: np.ndarray) -> pd.Series:
    nnz_per_row = np.count_nonzero(W, axis=1)
    return pd.Series(
        {
            "shape": str(W.shape),
            "dtype": str(W.dtype),
            "size_MB_dense": W.nbytes / 1024**2,
            "nnz": int(np.count_nonzero(W)),
            "density": float(np.count_nonzero(W) / W.size),
            "nnz_per_row_min": int(nnz_per_row.min()),
            "nnz_per_row_median": float(np.median(nnz_per_row)),
            "nnz_per_row_max": int(nnz_per_row.max()),
        }
    )

## Build one velocity trace from LAS

这是轻量版速度输入，只用于检查 `W_depth` 的实现。生产版建议把这里的 `velocity_trace_mps` 替换为 `lfm_depth.py` 生成的 `Vp_lfm[iline, xline, :]`。


In [ ]:
def load_well_vp_tvdss(
    las_path: Path,
    well_heads_file: Path,
    well_name: str,
    *,
    vp_mnemonic: str = "DTCO_QYZ_CLAER",
    vp_unit: str = "us/m",
) -> tuple[np.ndarray, np.ndarray, float]:
    las_file = lasio.read(las_path)
    vp_log = extract_vp_log_from_las(las_file, unit=vp_unit, curve_mnemonic=vp_mnemonic)

    heads = import_well_heads_petrel(well_heads_file).set_index("Name")
    if well_name not in heads.index:
        raise KeyError(f"{well_name!r} not found in well_heads.")
    kb_m = float(heads.loc[well_name, "Well datum value"])

    md_m = np.asarray(vp_log.basis, dtype=np.float64)
    tvdss_m = md_m - kb_m
    vp_mps = np.asarray(vp_log.values, dtype=np.float64)

    finite = np.isfinite(tvdss_m) & np.isfinite(vp_mps) & (vp_mps > 0.0)
    tvdss_m = tvdss_m[finite]
    vp_mps = vp_mps[finite]
    order = np.argsort(tvdss_m)
    return tvdss_m[order], vp_mps[order], kb_m


selected_las = LAS_DIR / f"{SELECTED_WELL}.las"
well_tvdss_m, well_vp_mps, kb_m = load_well_vp_tvdss(
    selected_las,
    WELL_HEADS_FILE,
    SELECTED_WELL,
    vp_mnemonic=VP_MNEMONIC,
    vp_unit=VP_UNIT,
)

depth_full_m = np.arange(
    GEOMETRY["sample_min"],
    GEOMETRY["sample_max"] + 0.5 * GEOMETRY["sample_step"],
    GEOMETRY["sample_step"],
    dtype=np.float64,
)
depth_mask = (
    (depth_full_m >= DEPTH_WINDOW_M[0])
    & (depth_full_m <= DEPTH_WINDOW_M[1])
    & (depth_full_m >= well_tvdss_m[0])
    & (depth_full_m <= well_tvdss_m[-1])
)
depth_demo_m = depth_full_m[depth_mask]
velocity_raw_mps = np.interp(depth_demo_m, well_tvdss_m, well_vp_mps)
sigma_samples = float(LOWPASS_SIGMA_M / GEOMETRY["sample_step"])
velocity_trace_mps = gaussian_filter1d(velocity_raw_mps, sigma=sigma_samples, mode="nearest")

print(f"well: {SELECTED_WELL}, KB={kb_m:.2f} m")
print(f"well TVDSS range: {well_tvdss_m[0]:.1f} to {well_tvdss_m[-1]:.1f} m")
print(f"W demo depth range: {depth_demo_m[0]:.1f} to {depth_demo_m[-1]:.1f} m, N={depth_demo_m.size}")
print(f"velocity range after smoothing: {np.nanmin(velocity_trace_mps):.1f} to {np.nanmax(velocity_trace_mps):.1f} m/s")

fig, ax = plt.subplots(figsize=(4.2, 5.0))
ax.plot(velocity_raw_mps, depth_demo_m, lw=0.8, alpha=0.45, label="resampled Vp")
ax.plot(velocity_trace_mps, depth_demo_m, lw=1.6, label=f"smoothed Vp, sigma={LOWPASS_SIGMA_M:.0f} m")
ax.invert_yaxis()
ax.set_xlabel("Vp (m/s)")
ax.set_ylabel("TVDSS depth (m)")
ax.grid(True, alpha=0.25)
ax.legend();

## Build `W_depth` and inspect its banded structure


In [ ]:
W_depth, W_meta = build_w_depth_dense(
    depth_demo_m,
    velocity_trace_mps,
    wavelet_time_s,
    wavelet_amp,
    amplitude_threshold=1e-7,
)

summary = summarize_w_depth(W_depth)
display(summary)

twt_sample_s = W_meta["twt_sample_s"]
print(f"TWT span in selected depth window: {twt_sample_s[-1] - twt_sample_s[0]:.3f} s")
print(f"Average depth sample interval in time: {np.median(np.diff(twt_sample_s)) * 1000:.2f} ms")

In [ ]:
vmax = float(np.nanmax(np.abs(W_depth)))
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2), constrained_layout=True)

im = axes[0].imshow(W_depth, aspect="auto", cmap="seismic", vmin=-vmax, vmax=vmax)
axes[0].set_title("W_depth dense matrix")
axes[0].set_xlabel("reflectivity interface index j")
axes[0].set_ylabel("seismic depth sample index l")
fig.colorbar(im, ax=axes[0], shrink=0.85)

nnz_per_row = np.count_nonzero(W_depth, axis=1)
axes[1].plot(nnz_per_row, depth_demo_m, lw=1.2)
axes[1].invert_yaxis()
axes[1].set_title("nonzero weights per output sample")
axes[1].set_xlabel("nnz per row")
axes[1].set_ylabel("TVDSS depth (m)")
axes[1].grid(True, alpha=0.25);

## Quick forward check

这里用 `AI ~= Vp` 的玩具假设，只验证矩阵乘法路径：`AI -> reflectivity -> W_depth @ r -> d_syn`。真实反演中 `AI` 来自网络输出或低频模型，不应简单等于速度。


In [ ]:
ai_demo = velocity_trace_mps.astype(np.float64)
r_demo = reflectivity_from_impedance(ai_demo)
d_syn_demo = W_depth @ r_demo

fig, axes = plt.subplots(1, 3, figsize=(10.0, 5.0), sharey=True, constrained_layout=True)
axes[0].plot(ai_demo, depth_demo_m, lw=1.2)
axes[0].set_title("AI demo")
axes[0].set_xlabel("AI proxy")
axes[0].set_ylabel("TVDSS depth (m)")

interface_depth_m = 0.5 * (depth_demo_m[:-1] + depth_demo_m[1:])
axes[1].plot(r_demo, interface_depth_m, lw=0.8)
axes[1].set_title("Reflectivity")
axes[1].set_xlabel("r")

axes[2].plot(d_syn_demo, depth_demo_m, lw=1.0)
axes[2].set_title("Synthetic depth trace")
axes[2].set_xlabel("amplitude")

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.25)

## Sparse storage

单道 dense 很小；批量训练时更推荐按 batch 构造或缓存 CSR/COO。全工区逐道保存 dense `W` 不现实。


In [ ]:
# W_csr = sparse.csr_matrix(W_depth)
# print(W_csr)
# print(f"dense MB: {W_depth.nbytes / 1024**2:.3f}")
# print(f"CSR data MB: {(W_csr.data.nbytes + W_csr.indices.nbytes + W_csr.indptr.nbytes) / 1024**2:.3f}")

# OUT_DIR = DATA / "_debug" / "w_depth_precompute_demo"
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# out_file = OUT_DIR / f"W_depth_demo_{SELECTED_WELL}.npz"

# # 保存 dense 示例，方便后续快速检查。生产版建议保存 CSR/COO 或按 batch 缓存。
# np.savez_compressed(
#     out_file,
#     W_depth=W_depth,
#     depth_m=depth_demo_m,
#     velocity_mps=velocity_trace_mps.astype(np.float32),
#     wavelet_time_s=wavelet_time_s.astype(np.float32),
#     wavelet_amp=wavelet_amp.astype(np.float32),
#     twt_sample_s=twt_sample_s.astype(np.float32),
# )
# out_file